#### Get TSC

In [1]:
import pandas as pd
import os
from NewsSentiment import TargetSentimentClassifier
from NewsSentiment.dataset import TooLongTextException
from openai import OpenAI
from tqdm.auto import tqdm
tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

gpt_api_key = os.environ.get('OPENAI_API_KEY')

tsc = TargetSentimentClassifier()

gpt_client = OpenAI(
    api_key=gpt_api_key
)

llama_client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

with open('prompt_EST.md', 'r', encoding='utf-8') as f:
    system_prompt_EST = f.read()

with open('prompt_OPP.md', 'r', encoding='utf-8') as f:
    system_prompt_OPP = f.read()

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
EST_datapoints = [
    ('Even as a post-poll combination was being worked out, Maharashtra Governor ', 'Bhagat Singh Koshyari', '’s controversial decision to administer the oath to Devendra Fadnavis of the Bharatiya Janata Party (BJP) as Chief Minister and Ajit Pawar of the Nationalist Congress Party (NCP) as Deputy Chief Minister was taken to court by the Shiv Sena, the NCP and the Congress.'),
    ('After the Supreme Court of India ordered an early floor test, ', 'Ajit Pawar', ' resigned.'),
    ('The ', 'BJP', ' and Prime Minister Narendra Modi, on Thursday, scripted history by becoming the second political party in the Indian history to come back to power, even stronger this time.'),
    ('While retaining its vote share in the Hindi Belt, the ', 'BJP', ' successfully made inroads in West Bengal, Odisha and Telangana.'),
    ('Moments after President Ram Nath Kovind accepted resignation of the Union Council of Ministers, Prime Minister ', 'Narendra Modi', ' said while the sun has set on the term of the present government, the'),
    ('The Supreme Court on Thursday gave the Election Commission of India (ECI) time till May 6 to take a final call on the complaints filed by the Congress against alleged hate speeches and misuse of the armed forces as political propaganda by Prime Minister ', 'Narendra Modi', ' and BJP president Amit Shah.'),
    ('As soon as the elections, in which Prime Minister ', 'Narendra Modi', ' will make his bid to return to power amid hectic parleys by several political parties to put a united fight against the ruling BJP, are announced, the model code of conduct will come into force.'),
    ('In the third phase, voting was held in 116 Lok Sabha seats, including all constituencies in Gujarat and Kerala, with ', 'BJP', ' president Amit Shah, Congress chief Rahul Gandhi and several Union Ministers among prominent candidates in the fray.'),
    ('The ', 'BJP', ' that has considerable presence in the region aims to improve its tally banking on the “Modi wave” and the support of dominant Lingayat community, considered as it’s vote base.'),
    ('The lone Muslim candidate from Gurugram, he is pitted against the Bharatiya Janata Party’s (', 'BJP', ') Rao Inderjit Singh and Congress’ Captain Ajay Yadav.'),
    ('Third, from the point of view of the ', 'BJP', ', we are looking at this election as a ‘gateway to the south’.'),
    ('But, as a political party, the ', 'BJP', ' also indulges in caste politics.'),
    ('Livelihood issues on a local level were more of a concern for voters and the main opposition ', 'BJP', ' was criticised for playing the "terror card".'),
    ("See a map of states voting With all the votes counted, Congress won Delhi by 42 seats to the Bharatiya Janata Party's 23 and captured Rajasthan, winning 96 seats to the ", 'BJP', "'s 78."),
    ('Speakers at the rally said both the Congress and the ', 'BJP', "-led alliances had failed to address people's grievances."),
    ('The ', 'BJP', ' said it would also build a million houses for the poor every year and reduce interest rates to make urban housing affordable.'),
    ('The ', 'BJP', "'s main rival is the governing Congress party."),
    ('Eligible voters: 714 million Polling centres: 828,804 Voting days: 16, 23, 30 April; 7, 13 May Vote counting: 16 May Leading candidates: Manmohan Singh (Congress), LK Advani (', 'BJP', '), Mayawati Live'),
    ('Congress leader Sonia Gandhi and ', 'BJP', ' prime ministerial candidate LK Advani were among candidates whose seats were holding voting in the third phase.'),
    ("State television says Congress's alliance has won or is ahead in 263 seats, compared with the ", 'BJP', "'s (154), the Third Front (60) and others (66)."),
    ('The ', 'BJP', ' had written to the Election Commission saying the government should withdraw the announcement of the plan until the polls were over in the two states.'),
    ('With an aim to connect with Indians living in foreign countries,the ', 'BJP', ' will expand its “Overseas Friends of BJP” cell in all states.'),
    ('A party leader said the overseas cell will promote and propagate policies and principles of the ', 'BJP', ' and strengthen support for the party among Indians living abroad.'),
    ('“They can influence their relatives in India to vote for and support the ', 'BJP', ' in elections,” Thaker said.'),
    ('While Pilibhit MP ', 'Varun Gandhi', ' has decided to contest from Sultanpur and leave'),

    ('', 'Bharatiya Janata Party', ' (BJP) president Rajnath Singh criticised the Congress party on Wednesday for the unprecedented growth in the level of poverty and unemployment.'),
    ('Jaipur/Bhopal Prime Minister Narendra Modi on Thursday accused the Congress of sympathising with terrorism and condoning corruption, and said that the ', 'BJP', '’s rise to power in the 2014 Lok Sabha elections broke the back of an ecosystem of middlemen who siphoned public money.'),
    ('In Chhattisgarh, the main contest is between the ruling Congress and the ', 'BJP', ', with the Aam Aadmi Party and BSP also in the fray.'),
    ('Chhattisgarh elections: Five seats where ', 'BJP', ' failed to make inroads'),
    ('Yediyurappa BS Yediyurappa, the veteran ', 'BJP', " leader, stated that he did not expect his son, BY Vijayendra, to be appointed as the party's Karnataka unit president."),
    ('Vijayendra expressed gratitude for the opportunity and pledged to work under the guidance of senior leaders to strengthen Prime Minister ', 'Narendra Modi', "'s position."),
    ('Pilot Congress leader Sachin Pilot has criticized the ', 'BJP', ' for trying to polarize voters on religious lines.'),
    ('He said that the ', 'BJP', ' is scared of defeat and is using central agencies to divert attention from the issue of development.'),
    ('But isn’t it odd that Congress should be facing anti-incumbency in Madhya Pradesh when it is ', 'BJP', ' which has been in office since 2003, barring a break of 1.5 years?'),
    ('In an interview with PTI on board a helicopter, while on the campaign trail in Madhya Pradesh, Mr. Pilot said all members of the INDIA bloc are committed to working together to defeat the ', 'BJP', ' in 2024 and who "will take what position will be decided after the polls".'),
    ('He praised the active participation of women in the election and acknowledged their role in prioritizing the return of the ', 'BJP', ' government.'),
    ('However, with general elections around the corner, both the ', 'BJP', ' and the Congress have a lot more at stake in MP and Chhattisgarh.'),
    ('BRS, Congress & INDIA bloc anti-Dalit, says PM in Telangana Prime Minister ', 'Narendra Modi', ' criticized the BSP, Congress, and JD(U) for being anti-Dalit during a rally in Hyderabad.'),
    ('Madhya Pradesh Polls: Voting For 230 Assembly Seats Begins, ', 'BJP', '-Congress In Close Contest'),
    ('All these have been turned upside down by the BJP, thanks to its indomitable leader, Mr ', 'Narendra Modi', '.'),
    ('Congress leader Priyanka Gandhi Vadra criticized the ', 'BJP', ' in Rajasthan, claiming that the party is scattered and lacks a chief ministerial face for the upcoming state Assembly polls.'),
    ('writing the Rules Mr Narendra Modi has re-written the rules of the ', 'BJP', ' and, to some extent, the rules of elections.'),
    ('', 'BJP', " expels Jitendra Meena for contesting election from Bassi against party's official candidate The Rajasthan BJP has expelled Jitendra Meena, former state president of the Scheduled Tribe Morcha, for contesting in the upcoming state election against the officially declared candidate of the party."),
    ('', 'BJP', " scattered in Rajasthan, doesn't have CM face:"),
    ('The party criticized the ', 'BJP', " for betraying the people's trust and neglecting the issues of the region."),
    ('The ', 'BJP', ' has already released its manifesto on November 16, which has pledged various welfare schemes for women, youth and students.'),
    ('The transformation of the Bharata Janata Party’s Hindutva politics under ', 'Narendra Modi', ' is multilayered, but one defining feature is its ability to subsume caste identities.'),
    ('As the polling process is underway in Rajasthan, ', 'BJP', " leader Alka Singh Gurjar criticized Rahul Gandhi's 'panauti' remark against Prime Minister Narendra Modi, stating that it reflects the values of Rahul Gandhi and the Congress party."),
    ('The Ladli Behna Yojana to give financial help to women has helped the ', 'BJP', ' in Madhya Pradesh.'),
    ('The ', 'BJP', ' spokesperson also criticized the Congress and the Communist Party of India (Marxist) for their silence on the participation of Hamas leader Khalid Mashal in a conference organized in Kerala.')
]

In [3]:
human_EST_labels = [
    'negative',
    'neutral',
    'positive',
    'positive',
    'neutral',
    'negative',
    'neutral',
    'neutral',
    'neutral',
    'neutral',
    'neutral',
    'negative',
    'negative',
    'neutral',
    'negative',
    'positive',
    'neutral',
    'neutral',
    'neutral',
    'neutral',
    'neutral',
    'positive',
    'positive',
    'positive',
    'neutral',

    'neutral',
    'positive',
    'neutral',
    'negative',
    'neutral',
    'positive',
    'negative',
    'negative',
    'neutral',
    'negative',
    'positive',
    'neutral',
    'neutral',
    'neutral',
    'positive',
    'negative',
    'neutral',
    'neutral',
    'negative',
    'negative',
    'positive',
    'neutral',
    'neutral',
    'positive',
    'neutral'    
]

In [4]:
OPP_datapoints = [
    ('A new post-poll combination, between the Sena, NCP and the ', 'Congress', ' and some independents, has now formed the government.'),
    ('The ', 'Congress', ' once again fell short of numbers needed to get Leader of Opposition position in Lok Sabha.'),
    ('In the third phase, voting was held in 116 Lok Sabha seats, including all constituencies in Gujarat and Kerala, with BJP president Amit Shah, ', 'Congress', ' chief Rahul Gandhi and several Union Ministers among prominent candidates in the fray.'),
    ('“The farzi dosti of bua and babua will also end the same way,” Modi said at an election rally here, referring to ', 'Mayawati', ' and SP president Akhilesh Yadav.'),
    ('The lone Muslim candidate from Gurugram, he is pitted against the Bharatiya Janata Party’s (BJP) Rao Inderjit Singh and ', 'Congress', '’ Captain Ajay Yadav.'),
    ('The ', 'Congress', ' appeared to make a comeback, forming the governments in the States after last year’s Assembly polls.'),
    ('The run-up to the election saw leaders from across the country — including Prime Minister Narendra Modi and ', 'Congress', ' president Rahul Gandhi — campaigning extensively.'),
    ('See a map of states voting With all the votes counted, ', 'Congress', " won Delhi by 42 seats to the Bharatiya Janata Party's 23 and captured Rajasthan, winning 96 seats to the BJP's 78."),
    ('In Mizoram in the north-east, ', 'Congress', ' swept back to power for the first time in a decade.'),
    ('The incumbent ', 'Congress', ' party-led coalition and parties led by the Hindu nationalist Bharatiya Janata Party will be battling a host of smaller parties.'),
    ('Speakers at the rally said both the ', 'Congress', " and the BJP-led alliances had failed to address people's grievances."),
    ("India's governing ", 'Congress', ' party has released its election manifesto, appealing for a return to power so it can continue to push economic growth.'),
    ('Congress president ', 'Sonia Gandhi', ' said "security and prosperity of all citizens will be our endeavour".'),
    ('Leading candidates: Manmohan Singh (', 'Congress', '), LK Advani (BJP), Mayawati ("Third front")'),
    ('', 'Mayawati', ' has ambitions to become a pan-Indian leader and could be a crucial player in'),
    ("The BJP's main rival is the governing ", 'Congress', ' party.'),
    ('Polls were held in 140 constituencies in 12 states, in a vote which pits the ', 'Congress', '-led coalition and opposition BJP-led bloc against smaller parties.'),
    ('The ruling ', 'Congress', '-led coalition and opposition BJP-led alliance are scrambling to gain pledges of support with a hung parliament predicted.'),
    ("The leaders of India's ", 'Congress', ' party have thanked the people for returning them to power with a "massive mandate".'),
    ('Congress President ', 'Sonia Gandhi', ' said that they had made the "right choice" and PM Manmohan Singh vowed the party would "rise to the occasion".'),
    ('', 'Congress', ' has made tackling the effects of the global economic crisis and ensuring growth its key priorities.'),
    ('The ', 'Congress', ' party is hoping to retain power in the three states'),
    ('In the northern state of Haryana, the ', 'Congress', ' is seeking a second term in office against a divided opposition, correspondents say.'),
    ('If this was odd, odder still was the ', 'Congress', ' second-in-command’s refusal to woo the electorate with the kind of largesse and promises mandatorily handed out ahead of a critically important general election.'),
    ('Sources said the party does not really want ', 'Mayawati', '’s dismissal; it wants her to stay on and destroy all chances of BSP’s reelection in 2012 by her misrule.'),

    ('Congress leader ', 'Rahul Gandhi', " spoke to farmers in Chhattisgarh and hailed his party's 'pro-farmer' model."),
    ('Vijayvargiya also targeted Congress leaders Kamal Nath, Priyanka Gandhi, and ', 'Rahul Gandhi', ', questioning their credibility.'),
    ('He urged ', 'Congress', ' to work for the poor and not divide society based on caste.'),
    ('Jaipur/Bhopal Prime Minister Narendra Modi on Thursday accused the ', 'Congress', ' of sympathising with terrorism and condoning corruption, and said that the BJP’s rise to power in the 2014 Lok Sabha elections broke the back of an ecosystem of middlemen who siphoned public money.'),
    ('Prime Minister Narendra Modi attacked Chief Minister Bhupesh Baghel over the alleged Mahadev betting app scam, while the ', 'Congress', ' focused on the welfare schemes of the Baghel government.'),
    ('In Chhattisgarh, the main contest is between the ruling ', 'Congress', ' and the BJP, with the Aam Aadmi Party and BSP also in the fray.'),
    ('The party failed to come to an agreement with the ', 'Congress', ' regarding seat sharing.'),
    ('', 'Congress', ' leader Jairam Ramesh criticized Prime Minister Modi, stating that he is the only Prime Minister in the world who can never tell the truth.'),
    ('Be it Prime Minister Narendra Modi’s announcement to extend the free ration scheme for five more years or Rahul Gandhi’s aggressive OBC pitch and demand for a caste census, both the BJP and ', 'Congress', ' have used the present round of Assembly elections to frame their respective 2024 Lok Sabha poll campaign narratives.'),
    ('Given that Preneet’s vote ensured that the Lok Sabha panel wasn’t tied over expulsion of Moitra, it is an embarrassment for the ', 'Congress', ' at a time when it is leading the Opposition unity efforts for the 2024 Lok Sabha elections.'),
    ('Samajwadi Party chief Akhilesh Yadav has hit out at Congress leader ', 'Rahul Gandhi', ' after the latter’s “X-ray" remark over the caste census demand, in yet another indication of the widening rift between the two INDIA opposition bloc members.'),
    ('Pilot ', 'Congress', ' leader Sachin Pilot has criticized the BJP for trying to polarize voters on religious lines.'),
    ('We also published the following articles recently ', 'Congress', ", BJP look for pole position in Chhindwara; will Kamal Nath trump 'kamal' in his backyard?"),
    ('Rao also accused the ', 'Congress', ' of using people, especially Muslims, as vote banks and questioned them over the demolition of Babri Masjid.'),
    ('', 'Congress', ' leader Priyanka Gandhi Vadra criticized the BJP in Rajasthan, claiming that the party is scattered and lacks a chief ministerial face for the upcoming state Assembly polls.'),
    ('', 'Congress', ' leader Jairam Ramesh criticized Prime Minister Modi, stating that he is the only Prime Minister in the world who can never tell the truth.'),
    ('In the 2019 parliamentary polls, in an embarrassing upset for the Congress, ', 'Rahul Gandhi', ' lost the Amethi seat to the BJP’s Smriti Irani by a margin of over 55,000 votes.'),
    ("He also highlighted the Congress' track record in Rajasthan and praised the ", 'Ashok Gehlot', ' government for fulfilling over 90% of the promises made in the previous manifesto.'),
    ('Punjab CM ', 'Bhagwant Mann', ' reiterates beginning citizen-centric services at doorstep'),
    ('On whether the BJP’s move not to project any CM face would benefit the ', 'Congress', ' in the polls, Gehlot said, “It naturally helps.'),
    ('Before setting out for his final round of campaigning, CM Gehlot, who is aiming to buck the three-decade trend of alternating governments and hoping for his fourth term, in an interview with The Indian Express, expressed confidence that the ', 'Congress', ' will return to power riding on his “shandaar” (excellent) performance, reports Hamza Khan.'),
    ("As the polling process is underway in Rajasthan, BJP leader Alka Singh Gurjar criticized Rahul Gandhi's 'panauti' remark against Prime Minister Narendra Modi, stating that it reflects the values of ", 'Rahul Gandhi', ' and the Congress party.'),
    ('The ', 'Mamata Banerjee', '-led party has accused the Centre of not releasing Central funds to the state because of political reasons and held protests in Delhi and Kolkata in October.'),
    ('Citing political violence, corruption, and appeasement in the state, the union minister said that Chief Minister ', 'Mamata Banerjee', ' has destroyed West Bengal.'),
    ('The ', 'Congress', ', buoyed by its success in Karnataka, is trying to pitch for maarpu (change) in Telangana.'),
    
]

In [5]:
human_OPP_labels = [
    'neutral',
    'negative',
    'neutral',
    'negative',
    'neutral',
    'positive',
    'neutral',
    'neutral',
    'positive',
    'neutral',
    'negative',
    'positive',
    'positive',
    'neutral',
    'positive',
    'neutral',
    'neutral',
    'negative',
    'positive',
    'positive',
    'positive',
    'neutral',
    'neutral',
    'negative',
    'negative',

    'positive',
    'negative',
    'negative',
    'negative',
    'positive',
    'neutral',
    'neutral',
    'neutral',
    'neutral',
    'negative',
    'negative',
    'neutral',
    'neutral',
    'negative',
    'neutral',
    'neutral',
    'negative',
    'positive',
    'positive',
    'neutral',
    'positive',
    'negative',
    'neutral',
    'negative',
    'neutral'

]

In [6]:
df_EST_annotations = pd.DataFrame({
    "Sentence": [datapoint[0] + datapoint[1] + datapoint[2] for datapoint in EST_datapoints],
    "Entity": [datapoint[1] for datapoint in EST_datapoints],
    "Human": human_EST_labels
})

df_EST_annotations

,Sentence,Entity,Human
0,Even as a post-poll combination was being work...,Bhagat Singh Koshyari,negative
1,After the Supreme Court of India ordered an ea...,Ajit Pawar,neutral
2,"The BJP and Prime Minister Narendra Modi, on T...",BJP,positive
3,While retaining its vote share in the Hindi Be...,BJP,positive
4,Moments after President Ram Nath Kovind accept...,Narendra Modi,neutral
5,The Supreme Court on Thursday gave the Electio...,Narendra Modi,negative
6,"As soon as the elections, in which Prime Minis...",Narendra Modi,neutral
7,"In the third phase, voting was held in 116 Lok...",BJP,neutral
8,The BJP that has considerable presence in the ...,BJP,neutral
9,"The lone Muslim candidate from Gurugram, he is...",BJP,neutral


In [7]:
df_OPP_annotations = pd.DataFrame({
    "Sentence": [datapoint[0] + datapoint[1] + datapoint[2] for datapoint in OPP_datapoints],
    "Entity": [datapoint[1] for datapoint in OPP_datapoints],
    "Human": human_OPP_labels
})

df_OPP_annotations

,Sentence,Entity,Human
0,"A new post-poll combination, between the Sena,...",Congress,neutral
1,The Congress once again fell short of numbers ...,Congress,negative
2,"In the third phase, voting was held in 116 Lok...",Congress,neutral
3,“The farzi dosti of bua and babua will also en...,Mayawati,negative
4,"The lone Muslim candidate from Gurugram, he is...",Congress,neutral
5,"The Congress appeared to make a comeback, form...",Congress,positive
6,The run-up to the election saw leaders from ac...,Congress,neutral
7,See a map of states voting With all the votes ...,Congress,neutral
8,"In Mizoram in the north-east, Congress swept b...",Congress,positive
9,The incumbent Congress party-led coalition and...,Congress,neutral


In [8]:
GRU_TSC_EST_labels = []
for target in tqdm(EST_datapoints):
    try:
        s = tsc.infer(targets=[target], batch_size=1, disable_tqdm=True)
        labels = [result[0]['class_label'] for result in s]
        GRU_TSC_EST_labels.extend(labels)
    except TooLongTextException:
        continue

GRU_TSC_OPP_labels = []
for target in tqdm(OPP_datapoints):
    try:
        s = tsc.infer(targets=[target], batch_size=1, disable_tqdm=True)
        labels = [result[0]['class_label'] for result in s]
        GRU_TSC_OPP_labels.extend(labels)
    except TooLongTextException:
        continue

100%|██████████| 50/50 [00:08<00:00,  5.65it/s]


In [9]:
df_EST_annotations['NewsSentiment'] = GRU_TSC_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment
0,Even as a post-poll combination was being work...,Bhagat Singh Koshyari,negative,negative
1,After the Supreme Court of India ordered an ea...,Ajit Pawar,neutral,negative
2,"The BJP and Prime Minister Narendra Modi, on T...",BJP,positive,positive
3,While retaining its vote share in the Hindi Be...,BJP,positive,positive
4,Moments after President Ram Nath Kovind accept...,Narendra Modi,neutral,neutral
5,The Supreme Court on Thursday gave the Electio...,Narendra Modi,negative,negative
6,"As soon as the elections, in which Prime Minis...",Narendra Modi,neutral,positive
7,"In the third phase, voting was held in 116 Lok...",BJP,neutral,neutral
8,The BJP that has considerable presence in the ...,BJP,neutral,positive
9,"The lone Muslim candidate from Gurugram, he is...",BJP,neutral,neutral


In [10]:
df_OPP_annotations['NewsSentiment'] = GRU_TSC_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment
0,"A new post-poll combination, between the Sena,...",Congress,neutral,neutral
1,The Congress once again fell short of numbers ...,Congress,negative,negative
2,"In the third phase, voting was held in 116 Lok...",Congress,neutral,neutral
3,“The farzi dosti of bua and babua will also en...,Mayawati,negative,neutral
4,"The lone Muslim candidate from Gurugram, he is...",Congress,neutral,neutral
5,"The Congress appeared to make a comeback, form...",Congress,positive,positive
6,The run-up to the election saw leaders from ac...,Congress,neutral,neutral
7,See a map of states voting With all the votes ...,Congress,neutral,positive
8,"In Mizoram in the north-east, Congress swept b...",Congress,positive,positive
9,The incumbent Congress party-led coalition and...,Congress,neutral,neutral


In [11]:
def LLM_annotate(datapoint, sys_prompt, model_name):
    left, entity, right = datapoint
    sentence = left + entity + right

    input = "Sentence: " + sentence + "\nEntity: " + entity

    if "gpt" in model_name:
        chat_completion = gpt_client.chat.completions.create(
            messages=[
                {
                    'role': 'system',
                    'content': sys_prompt,
                },
                {
                    'role': 'user',
                    'content': input,
                }
            ],
            model=model_name,
            seed=42, # temperature unavaliable for gpt5+ models
            reasoning_effort="none" # as per industry practice: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5
        )

    elif "llama" in model_name:
        chat_completion = llama_client.chat.completions.create(
            messages=[
                {
                    'role': 'system',
                    'content': sys_prompt,
                },
                {
                    'role': 'user',
                    'content': input,
                }
            ],
            model=model_name,
            temperature=0,
            reasoning_effort="none" # as per industry practice: https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning?tabs=csharp%2Cgpt-5
        )
        
        
    return chat_completion.choices[0].message.content

In [12]:
gpt_EST_labels = []
for datapoint in tqdm(EST_datapoints):
    gpt_EST_labels.append(LLM_annotate(datapoint, system_prompt_EST, "gpt-5.6-luna"))

gpt_OPP_labels = []
for datapoint in tqdm(OPP_datapoints):
    gpt_OPP_labels.append(LLM_annotate(datapoint, system_prompt_OPP, "gpt-5.6-luna"))

100%|██████████| 50/50 [01:00<00:00,  1.21s/it]


In [13]:
df_EST_annotations['ChatGPT'] = gpt_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT
0,Even as a post-poll combination was being work...,Bhagat Singh Koshyari,negative,negative,negative
1,After the Supreme Court of India ordered an ea...,Ajit Pawar,neutral,negative,neutral
2,"The BJP and Prime Minister Narendra Modi, on T...",BJP,positive,positive,positive
3,While retaining its vote share in the Hindi Be...,BJP,positive,positive,positive
4,Moments after President Ram Nath Kovind accept...,Narendra Modi,neutral,neutral,neutral
5,The Supreme Court on Thursday gave the Electio...,Narendra Modi,negative,negative,negative
6,"As soon as the elections, in which Prime Minis...",Narendra Modi,neutral,positive,neutral
7,"In the third phase, voting was held in 116 Lok...",BJP,neutral,neutral,neutral
8,The BJP that has considerable presence in the ...,BJP,neutral,positive,positive
9,"The lone Muslim candidate from Gurugram, he is...",BJP,neutral,neutral,neutral


In [14]:
df_OPP_annotations['ChatGPT'] = gpt_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT
0,"A new post-poll combination, between the Sena,...",Congress,neutral,neutral,neutral
1,The Congress once again fell short of numbers ...,Congress,negative,negative,negative
2,"In the third phase, voting was held in 116 Lok...",Congress,neutral,neutral,neutral
3,“The farzi dosti of bua and babua will also en...,Mayawati,negative,neutral,negative
4,"The lone Muslim candidate from Gurugram, he is...",Congress,neutral,neutral,neutral
5,"The Congress appeared to make a comeback, form...",Congress,positive,positive,positive
6,The run-up to the election saw leaders from ac...,Congress,neutral,neutral,neutral
7,See a map of states voting With all the votes ...,Congress,neutral,positive,positive
8,"In Mizoram in the north-east, Congress swept b...",Congress,positive,positive,positive
9,The incumbent Congress party-led coalition and...,Congress,neutral,neutral,neutral


In [15]:
llama_EST_labels = []
for datapoint in tqdm(EST_datapoints):
    llama_EST_labels.append(LLM_annotate(datapoint, system_prompt_EST, "llama3.1:8b"))

llama_OPP_labels = []
for datapoint in tqdm(OPP_datapoints):
    llama_OPP_labels.append(LLM_annotate(datapoint, system_prompt_OPP, "llama3.1:8b"))

100%|██████████| 50/50 [00:18<00:00,  2.64it/s]


In [16]:
df_EST_annotations['Llama'] = llama_EST_labels

df_EST_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
0,Even as a post-poll combination was being work...,Bhagat Singh Koshyari,negative,negative,negative,neutral
1,After the Supreme Court of India ordered an ea...,Ajit Pawar,neutral,negative,neutral,negative
2,"The BJP and Prime Minister Narendra Modi, on T...",BJP,positive,positive,positive,positive
3,While retaining its vote share in the Hindi Be...,BJP,positive,positive,positive,positive
4,Moments after President Ram Nath Kovind accept...,Narendra Modi,neutral,neutral,neutral,positive
5,The Supreme Court on Thursday gave the Electio...,Narendra Modi,negative,negative,negative,negative
6,"As soon as the elections, in which Prime Minis...",Narendra Modi,neutral,positive,neutral,negative
7,"In the third phase, voting was held in 116 Lok...",BJP,neutral,neutral,neutral,positive
8,The BJP that has considerable presence in the ...,BJP,neutral,positive,positive,positive
9,"The lone Muslim candidate from Gurugram, he is...",BJP,neutral,neutral,neutral,negative


In [17]:
df_OPP_annotations['Llama'] = llama_OPP_labels

df_OPP_annotations

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
0,"A new post-poll combination, between the Sena,...",Congress,neutral,neutral,neutral,neutral
1,The Congress once again fell short of numbers ...,Congress,negative,negative,negative,negative
2,"In the third phase, voting was held in 116 Lok...",Congress,neutral,neutral,neutral,neutral
3,“The farzi dosti of bua and babua will also en...,Mayawati,negative,neutral,negative,negative
4,"The lone Muslim candidate from Gurugram, he is...",Congress,neutral,neutral,neutral,neutral
5,"The Congress appeared to make a comeback, form...",Congress,positive,positive,positive,positive
6,The run-up to the election saw leaders from ac...,Congress,neutral,neutral,neutral,neutral
7,See a map of states voting With all the votes ...,Congress,neutral,positive,positive,positive
8,"In Mizoram in the north-east, Congress swept b...",Congress,positive,positive,positive,positive
9,The incumbent Congress party-led coalition and...,Congress,neutral,neutral,neutral,neutral


In [18]:
df_EST_annotations.to_csv("EST_tone_annotation.csv", index=False)
df_OPP_annotations.to_csv("OPP_tone_annotation.csv", index=False)

### Results

In [19]:
import pandas as pd

df_EST_annotations = pd.read_csv("EST_tone_annotation.csv")
df_OPP_annotations = pd.read_csv("OPP_tone_annotation.csv")

EST

In [20]:
(df_EST_annotations["Human"] == df_EST_annotations["NewsSentiment"]).sum()

42

In [21]:
(df_EST_annotations["Human"] == df_EST_annotations["ChatGPT"]).sum()

39

In [22]:
(df_EST_annotations["Human"] == df_EST_annotations["Llama"]).sum()

27

OPP

In [23]:
(df_OPP_annotations["Human"] == df_OPP_annotations["NewsSentiment"]).sum()

43

In [24]:
(df_OPP_annotations["Human"] == df_OPP_annotations["ChatGPT"]).sum()

37

In [25]:
(df_OPP_annotations["Human"] == df_OPP_annotations["Llama"]).sum()

33

In [26]:
df_OPP_annotations[df_OPP_annotations["NewsSentiment"] != df_OPP_annotations["ChatGPT"]]

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
3,“The farzi dosti of bua and babua will also en...,Mayawati,negative,neutral,negative,negative
22,"In the northern state of Haryana, the Congress...",Congress,neutral,neutral,positive,neutral
31,The party failed to come to an agreement with ...,Congress,neutral,neutral,negative,negative
32,Congress leader Jairam Ramesh criticized Prime...,Congress,neutral,neutral,positive,negative
36,Pilot Congress leader Sachin Pilot has critici...,Congress,neutral,neutral,positive,negative
39,Congress leader Priyanka Gandhi Vadra criticiz...,Congress,neutral,neutral,positive,negative
40,Congress leader Jairam Ramesh criticized Prime...,Congress,neutral,neutral,positive,negative
44,On whether the BJP’s move not to project any C...,Congress,neutral,neutral,positive,positive


In [27]:
df_EST_annotations[df_EST_annotations["NewsSentiment"] != df_EST_annotations["ChatGPT"]]

,Sentence,Entity,Human,NewsSentiment,ChatGPT,Llama
1,After the Supreme Court of India ordered an ea...,Ajit Pawar,neutral,negative,neutral,negative
6,"As soon as the elections, in which Prime Minis...",Narendra Modi,neutral,positive,neutral,negative
10,"Third, from the point of view of the BJP, we a...",BJP,neutral,neutral,positive,positive
13,See a map of states voting With all the votes ...,BJP,neutral,neutral,negative,negative
19,State television says Congress's alliance has ...,BJP,neutral,neutral,negative,negative
25,Bharatiya Janata Party (BJP) president Rajnath...,Bharatiya Janata Party,neutral,neutral,positive,negative
26,Jaipur/Bhopal Prime Minister Narendra Modi on ...,BJP,positive,negative,positive,positive
37,"BRS, Congress & INDIA bloc anti-Dalit, says PM...",Narendra Modi,neutral,neutral,negative,positive
41,writing the Rules Mr Narendra Modi has re-writ...,BJP,neutral,neutral,positive,negative
43,"BJP scattered in Rajasthan, doesn't have CM face:",BJP,negative,neutral,negative,negative
